# NYC Airbnb Open Data — Mini Project 1

Dataset: [dgomonov/new-york-city-airbnb-open-data](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) (`AB_NYC_2019.csv`)

Setup: `uv sync`, then select the `.venv` kernel. The dashboard lives in `app.py`.

**How to use this notebook:** every section maps to a project requirement. Starter code runs end-to-end; cells marked ✍️ are decisions or write-ups your team needs to own. Once a decision is final here, copy it into `app.py`.

## ✍️ Central question

_Pick ONE question every chart and widget serves. Suggestion: **"How much do borough and room type drive the nightly price of an NYC Airbnb?"**_

Our question: ...

## 0. Setup

In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

sns.set_theme()
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42

## 1. Data Cleaning (Pandas)

Requirements:
- [ ] Load the dataset
- [ ] Identify + handle missing values, **justify** drop vs. impute (% of rows affected, is missingness correlated with the outcome?)
- [ ] At least one `.groupby()` with an aggregation
- [ ] At least one derived column (`.str`, `.apply`, or `pd.to_datetime`)

### 1.1 Load

In [ ]:
# Download latest version (cached after first run)
path = Path(kagglehub.dataset_download('dgomonov/new-york-city-airbnb-open-data'))
print('Path to dataset files:', path)

df_raw = pd.read_csv(path / 'AB_NYC_2019.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

### 1.2 Missing values

In [ ]:
missing = pd.DataFrame({
    'n_missing': df_raw.isna().sum(),
    'pct_missing': (df_raw.isna().mean() * 100).round(2),
}).query('n_missing > 0').sort_values('pct_missing', ascending=False)
missing

In [ ]:
# Is missingness structured? Check whether the review columns go missing together, and why.
no_review = df_raw['last_review'].isna()
print('last_review and reviews_per_month missing on the same rows:', (no_review == df_raw['reviews_per_month'].isna()).all())
print('number_of_reviews on those rows:', df_raw.loc[no_review, 'number_of_reviews'].unique())

# Is missingness correlated with the outcome (price)?
df_raw.assign(has_review=~no_review).groupby('has_review')['price'].describe()

✍️ **Our missing-value decisions** (one line per column, with the % affected and why we dropped or imputed):

- `reviews_per_month` (…%): …
- `last_review` (…%): …
- `name`, `host_name` (…%): …
- Does missingness look correlated with price? …

In [ ]:
df = df_raw.copy()

# Suggested starting point -- change it if your justification above says otherwise.
# reviews_per_month is missing exactly when a listing has 0 reviews, so 0 is the true value.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# name / host_name: well under 1% missing and not model features, so keep the rows.
df[['name', 'host_name']] = df[['name', 'host_name']].fillna('Unknown')

# last_review: left as missing here; it becomes a derived column in 1.5.
df.isna().sum()

### 1.3 Strange values, inconsistencies, duplicates

These checks feed the "Faithfulness & Granularity" write-up in Section 2.

In [ ]:
print('Duplicate listing ids:', df['id'].duplicated().sum())
print('Fully duplicated rows:', df.duplicated().sum())
print('Listings priced at $0:', (df['price'] == 0).sum())
print('Listings over $1,000/night:', (df['price'] > 1000).sum())
print('Minimum nights > 365:', (df['minimum_nights'] > 365).sum())
df[['price', 'minimum_nights', 'number_of_reviews', 'availability_365']].describe()

✍️ **Our filtering decisions:** which rows did we remove and why? (e.g. $0 prices are impossible; extreme prices dominate an OLS fit; is a 1,250-night minimum still a short-term rental?)

In [ ]:
# Adjust these thresholds to match your justification above.
PRICE_MIN, PRICE_MAX = 1, 1000
MAX_MIN_NIGHTS = 365

before = len(df)
df = df[df['price'].between(PRICE_MIN, PRICE_MAX) & (df['minimum_nights'] <= MAX_MIN_NIGHTS)]
print(f'Dropped {before - len(df):,} rows ({(before - len(df)) / before:.1%}); {len(df):,} remain')

### 1.4 Group-by aggregation

In [ ]:
borough_room = (
    df.groupby(['neighbourhood_group', 'room_type'])['price']
    .agg(['count', 'median', 'mean'])
    .round(1)
)
borough_room

✍️ What does this table say about our central question? Is median or mean the better summary here, and why?

### 1.5 Derived / engineered columns

In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'])
snapshot_date = df['last_review'].max()  # most recent review ~ when the data was scraped
df['days_since_review'] = (snapshot_date - df['last_review']).dt.days
df['has_reviews'] = df['number_of_reviews'] > 0
df['name_length'] = df['name'].str.len()
df['log_price'] = np.log(df['price'])

df[['last_review', 'days_since_review', 'has_reviews', 'name_length', 'log_price']].head()

✍️ Optional: add a feature that serves *your* question, e.g. `df['name'].str.contains('luxury', case=False)`, or distance to Times Square with `.apply`.

## 2. EDA: Faithfulness & Granularity

This text goes in the dashboard or README. Helper facts below.

In [ ]:
print('Rows:', len(df_raw), '| unique listing ids:', df_raw['id'].nunique(), '| unique hosts:', df_raw['host_id'].nunique())
print('Last-review dates:', df['last_review'].min().date(), 'to', df['last_review'].max().date())
print('Boroughs:', df['neighbourhood_group'].nunique(), '| neighbourhoods:', df['neighbourhood'].nunique())
print('Hosts with 10+ listings:', df.loc[df['calculated_host_listings_count'] >= 10, 'host_id'].nunique())
df['neighbourhood_group'].value_counts()

✍️ **Faithfulness & Granularity**

- **What does each row represent?** …
- **What is the scope?** (place, time period, which listings are included or missing) …
- **Strange values, inconsistencies, duplicates we considered:** (refer to 1.3: $0 prices, extreme prices/minimum nights, multi-listing hosts, `price` is the *asking* price not what guests actually paid) …

## 3. Visualizations

Requirements (each must go into the dashboard via `st.pyplot`):
- [ ] Categorical distribution (bar plot)
- [ ] Quantitative distribution (histogram)
- [ ] Relationship between two variables (scatter or line)
- [ ] Bonus: correlation heatmap

### 3.1 Categorical distribution: bar plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='neighbourhood_group', hue='room_type', ax=ax,
              order=df['neighbourhood_group'].value_counts().index)
ax.set(title='Listings by borough and room type', xlabel='Borough', ylabel='Number of listings')
plt.show()

### 3.2 Quantitative distribution: histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price'], bins=50, ax=axes[0])
axes[0].set(title='Nightly price', xlabel='Price ($)')
sns.histplot(df['log_price'], bins=50, ax=axes[1])
axes[1].set(title='log(price)', xlabel='log(price)')
plt.show()

✍️ What shape is the price distribution? Does that change whether we should model `price` or `log_price`?

### 3.3 Relationship between two variables: scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(df['longitude'], df['latitude'], c=df['log_price'], s=2, alpha=0.5, cmap='viridis')
fig.colorbar(sc, ax=ax, label='log(price)')
ax.set(title='Where are the expensive listings?', xlabel='Longitude', ylabel='Latitude')
plt.show()

### 3.4 Bonus: correlation heatmap

In [ ]:
num_cols = ['price', 'log_price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
            'calculated_host_listings_count', 'availability_365', 'days_since_review']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation between numeric features')
plt.show()

✍️ Which numeric features are (and aren't) correlated with price? This tells you how much the model will need the categorical features.

## 4. One-Hot Encoding

Requirements:
- [ ] At least one categorical feature one-hot encoded (`pd.get_dummies`)
- [ ] Drop one category as the reference level (`drop_first=True`) and **state which one**
- [ ] Interpret one encoded coefficient relative to the reference (Section 5)
- [ ] Train and test end up with the **same** encoded columns
- [ ] Compare test error with vs. without the encoded feature (Section 5)

We split **before** encoding so the test set stays unseen. Fixing each column's categories from the training set means `drop_first` always drops the same reference level, and train/test always get identical columns, even if a rare category is missing from one split.

In [ ]:
TARGET = 'price'  # or 'log_price': decide and justify
CATEGORICAL = ['neighbourhood_group', 'room_type']
NUMERIC = ['minimum_nights', 'number_of_reviews', 'availability_365', 'calculated_host_listings_count']

train, test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')

# Categories come from the TRAINING set only. The first (alphabetical) one becomes the reference level.
categories = {c: sorted(train[c].unique()) for c in CATEGORICAL}

def make_features(data):
    cats = pd.DataFrame({c: pd.Categorical(data[c], categories=categories[c]) for c in CATEGORICAL}, index=data.index)
    dummies = pd.get_dummies(cats, drop_first=True, dtype=int)
    return pd.concat([data[NUMERIC], dummies], axis=1)

X_train, X_test = make_features(train), make_features(test)
y_train, y_test = train[TARGET], test[TARGET]

assert list(X_train.columns) == list(X_test.columns)
print('Reference levels (dropped):', {c: cats[0] for c, cats in categories.items()})
X_train.head()

✍️ **Reference levels we dropped:** … (why is having an intercept *plus* every dummy column a problem?)

## 5. A Simple Model

Requirements:
- [ ] Numeric target
- [ ] Constant model + one step up (OLS/MLR) that uses the one-hot features
- [ ] Predictions vs. actual plot (goes in the dashboard)
- [ ] State the train/test split + one evaluation observation

In [ ]:
def evaluate(name, y_true, y_pred):
    return {'model': name, 'RMSE': root_mean_squared_error(y_true, y_pred), 'MAE': mean_absolute_error(y_true, y_pred)}

# Constant model: always predict the training mean
const_pred = np.full(len(y_test), y_train.mean())

# OLS without the encoded features (numeric only)
ols_numeric = LinearRegression().fit(X_train[NUMERIC], y_train)

# OLS with numeric + one-hot encoded features
ols_full = LinearRegression().fit(X_train, y_train)

# Note: if TARGET is log_price, these errors are in log units, not dollars.
results = pd.DataFrame([
    evaluate('Constant (train mean)', y_test, const_pred),
    evaluate('OLS: numeric only', y_test, ols_numeric.predict(X_test[NUMERIC])),
    evaluate('OLS: numeric + one-hot', y_test, ols_full.predict(X_test)),
]).round(2)
results

In [ ]:
print('Intercept:', round(ols_full.intercept_, 2))
pd.Series(ols_full.coef_, index=X_train.columns).sort_values().round(2)

In [ ]:
test_pred = ols_full.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, test_pred, s=4, alpha=0.3)
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
ax.plot(lims, lims, 'r--', label='perfect prediction')
ax.set(xlabel=f'Actual {TARGET}', ylabel=f'Predicted {TARGET}', title='Predicted vs. actual (test set)')
ax.legend()
plt.show()

✍️ **Model write-up**

- **Train/test split:** 80/20 random split (`random_state=42`), … rows train / … rows test.
- **Coefficient interpretation** (plain language, relative to the reference level): "Listings in ___ are predicted to cost about $___ more/less than listings in ___, holding other features constant."
- **Did one-hot encoding help?** Compare the two OLS rows in the results table: …
- **Evaluation observation:** (look at the plot: where does the model miss, e.g. underpredicting at the high end?) …

## 6–7. Dashboard + Deployment → `app.py`

- [ ] Port the final cleaning decisions from Sections 1.2–1.5 into `load_data()` in `app.py`
- [ ] At least 2 widgets that change what's displayed (`app.py` has a borough multiselect, room-type selectbox and price slider)
- [ ] At least one `st.columns` layout (summary metrics row)
- [ ] Every chart via `st.pyplot`; Faithfulness & Granularity text + model write-up filled in
- [ ] Run locally: `uv run streamlit run app.py`
- [ ] Regenerate `requirements.txt` after any `uv add` (command in README), push to GitHub
- [ ] Deploy on [Streamlit Community Cloud](https://share.streamlit.io) (main file `app.py`, Python 3.12 under Advanced settings)
- [ ] Submit live link + repo link

## 8. Presentation notes (GM, Mon Oct 19)

Keep Julia's slide-decking prof-dev in mind. A few slides plus a live demo.

✍️ **Process:** central question → raw data → dashboard; key choices (cleaning, features, model) …

✍️ **Issues along the way:** messy data, deployment errors, model surprises, and how we got past them …

✍️ **Future expansion:** more data, better model, new features, new widgets …